# 05 用户流失预测模型

三模型对比训练：Logistic Regression / Random Forest / XGBoost
- SMOTE 过采样处理类别不平衡
- 5折分层交叉验证
- ROC 曲线、混淆矩阵、特征重要性分析

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42
BASE_DIR = Path('..').resolve()

## 5.1 加载数据与特征工程

In [ ]:
# 加载用户特征数据
df = pd.read_parquet(BASE_DIR / 'data' / 'processed' / 'user_features.parquet')
print(f"数据形状: {df.shape}")
print(f"流失用户占比: {df['is_churned'].mean():.2%}")

# 排除非特征列
exclude_cols = []
id_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['id', 'customer_unique'])]
date_cols = [c for c in df.columns if df[c].dtype == 'datetime64[ns]' or 'date' in c.lower()]
label_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['segment', 'label', 'cluster', 'group'])]
exclude_cols = list(set(id_cols + date_cols + label_cols + ['is_churned']))

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"\n特征列数: {len(feature_cols)}")

X = df[feature_cols].copy()
y = df['is_churned'].copy()

# 处理分类特征
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('unknown').astype(str))

X = X.fillna(0)
print(f"最终特征数: {X.shape[1]}")
X.head()

## 5.2 数据集划分与 SMOTE 过采样

In [ ]:
# 分层划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"训练集: {X_train.shape[0]} 样本, 流失率: {y_train.mean():.2%}")
print(f"测试集: {X_test.shape[0]} 样本, 流失率: {y_test.mean():.2%}")

# SMOTE 过采样（仅训练集）
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"\nSMOTE 前: {len(y_train)} 样本 (正: {y_train.sum()}, 负: {(~y_train.astype(bool)).sum()})")
print(f"SMOTE 后: {len(y_train_res)} 样本 (正: {y_train_res.sum()}, 负: {(~y_train_res.astype(bool)).sum()})")

## 5.3 模型定义与 5 折交叉验证

In [ ]:
# 定义三个模型
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')
}

# 5折交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    print(f"\n{'─' * 40}")
    print(f"交叉验证: {name}")
    scores = {}
    for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        cv_scores = cross_val_score(model, X_train_res, y_train_res, cv=skf, scoring=metric, n_jobs=-1)
        scores[metric] = {'mean': cv_scores.mean(), 'std': cv_scores.std()}
        print(f"  {metric}: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    cv_results[name] = scores

## 5.4 测试集评估

In [ ]:
# 测试集评估
test_results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_prob)
    }
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    
    test_results[name] = {
        'metrics': metrics, 'fpr': fpr, 'tpr': tpr, 'cm': cm,
        'y_pred': y_pred, 'y_prob': y_prob
    }
    
    print(f"\n{'─' * 40}")
    print(f"{name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

# 模型对比表
comparison = pd.DataFrame({name: r['metrics'] for name, r in test_results.items()}).T
print("\n=== 模型对比 ===")
comparison

## 5.5 ROC 曲线对比

In [ ]:
# ROC 曲线对比
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2196F3', '#4CAF50', '#FF5722']

for (name, result), color in zip(test_results.items(), colors):
    ax.plot(result['fpr'], result['tpr'], color=color, lw=2,
            label=f"{name} (AUC={result['metrics']['auc_roc']:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='随机基线')
ax.set_xlabel('假阳性率 (FPR)', fontsize=12)
ax.set_ylabel('真阳性率 (TPR)', fontsize=12)
ax.set_title('ROC 曲线对比 - 流失预测模型', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.6 混淆矩阵

In [ ]:
# 混淆矩阵
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, result) in zip(axes, test_results.items()):
    sns.heatmap(result['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['未流失', '已流失'], yticklabels=['未流失', '已流失'])
    ax.set_title(f'{name}\n混淆矩阵', fontsize=12)
    ax.set_xlabel('预测标签')
    ax.set_ylabel('真实标签')

plt.suptitle('各模型混淆矩阵对比', fontsize=14)
plt.tight_layout()
plt.show()

## 5.7 特征重要性

In [ ]:
# 特征重要性 Top 15
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
feature_names = X.columns.tolist()

for ax, (name, model) in zip(axes, models.items()):
    if name == 'Logistic Regression':
        importances = np.abs(model.coef_[0])
    else:
        importances = model.feature_importances_
    
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(15)
    
    sns.barplot(data=fi_df, x='importance', y='feature', ax=ax, palette='viridis')
    ax.set_title(f'{name}\n特征重要性 Top15', fontsize=12)
    ax.set_xlabel('重要性')
    ax.set_ylabel('')

plt.suptitle('各模型特征重要性对比', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 小结

- 使用 SMOTE 过采样解决了正负样本不平衡问题
- 三模型中 XGBoost/Random Forest 通常表现最优
- 关键流失特征：recency_days、total_orders、delayed_order_ratio 等
- 模型可用于识别高流失风险用户，指导精准召回策略